In [ ]:
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)

In [ ]:
#read the filtered object
adata=anndata.read('/g/korbel/olisov/Visium_Linda/adata_objects/11samples_normalized.h5ad')

In [ ]:
adata1=adata.copy()
seed=1
sc.tl.pca(adata1)
sc.pp.neighbors(adata1)
sc.tl.umap(adata1)
sc.pl.umap(adata1, color=['sample'], wspace=1)

In [ ]:
#set colour map
samples = list(adata1.obs['sample'].astype(str).unique())
adata1.obs['sample'] = adata1.obs['sample'].astype('category')
adata1.obs['sample'] = adata1.obs['sample'].cat.set_categories(samples, ordered=True)
colors = plt.cm.tab20(range(len(samples)))
hex_colors = [mcolors.to_hex(c) for c in colors]
adata1.uns['sample_colors'] = hex_colors

In [ ]:
#set dimensions
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8

with plt.rc_context({
    "figure.figsize": (fig_width_inch, fig_height_inch),
    "figure.dpi": 600,
    "font.size": 5,
    "axes.labelsize": 5}):
    ax = sc.pl.umap(adata1, color=['sample'], wspace=1, show=False, size=1, legend_loc=None)

    # Remove the title
    ax.set_title('')

    # Save the figure
    plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/11samples/umap.svg', dpi=600)
    plt.show()

In [ ]:
#umap for stroma spots only

In [ ]:
#read annotation
annot_dir = '/g/korbel/olisov/Visium_Linda/Spot_Annotation/'
files = [f for f in os.listdir(annot_dir) if f.endswith('.csv')]
annot_list = []
for file in files:
    path = os.path.join(annot_dir, file)
    df = pd.read_csv(path, names=['Barcode', 'Annotation'], skiprows=1)
    sample = file.split('_')[0]
    df['Barcode'] = sample + '_' + df['Barcode'].astype(str)
    annot_list.append(df)
annot = pd.concat(annot_list, ignore_index=True)
annot=annot[annot['Annotation']=='Stroma']
# Set Barcode as the index
annot.set_index('Barcode', inplace=True)

In [ ]:
# Merging the dataframes, adding the columns from annot to adata.obs
adata.obs = adata.obs.merge(annot, left_on='Barcode', right_index=True, how='left')
adata = adata[~adata.obs['Annotation'].isna()].copy()

In [ ]:
adata1=adata.copy()
seed=1
sc.tl.pca(adata1)
sc.pp.neighbors(adata1)
sc.tl.umap(adata1)s
sc.pl.umap(adata1, color=['sample'], wspace=1)

In [ ]:
#set colour map
samples = list(adata1.obs['sample'].astype(str).unique())
adata1.obs['sample'] = adata1.obs['sample'].astype('category')
adata1.obs['sample'] = adata1.obs['sample'].cat.set_categories(samples, ordered=True)
colors = plt.cm.tab20(range(len(samples)))
hex_colors = [mcolors.to_hex(c) for c in colors]
adata1.uns['sample_colors'] = hex_colors

In [ ]:
with plt.rc_context({
    "figure.figsize": (fig_width_inch, fig_height_inch),
    "figure.dpi": 600,
    "font.size": 5,
    "axes.labelsize": 5}):
    ax = sc.pl.umap(adata1, color=['sample'], wspace=1, show=False, size=2, legend_loc=None)

    # Remove the title
    ax.set_title('')

    # Save the figure
    plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/11samples/umap_stroma.svg', dpi=600)
    plt.show()

In [ ]:
#subset adata  by sample 2023-12-22_lane1LM4445_count
from cell2location.utils import select_slide
adata= select_slide(adata, '2023-12-22_lane1LM4445_count')

In [ ]:
# Function to extract and clean the sample name
def extract_sample_name(sample):
    parts = sample.split('_')
    sample_name = parts[1]
    cleaned_sample_name = sample_name.replace('lane1', '')
    return cleaned_sample_name
adata.obs['sample'] = adata.obs['sample'].apply(extract_sample_name)

In [ ]:
adata1=adata.copy()

In [ ]:
import random
random.seed(5)
sc.tl.pca(adata1)
sc.pp.neighbors(adata1)
sc.tl.umap(adata1)

In [ ]:
sc.tl.leiden(adata1, resolution=0.5)
sc.pl.umap(adata1, color=['leiden'], wspace=1)

In [ ]:
# Make sure 'leiden' is categorical
adata1.obs['leiden'] = adata1.obs['leiden'].astype('category')

# Rename categories
adata1.obs['leiden'].cat.categories = [f'Leiden {i + 1}' for i in range(len(adata1.obs['leiden'].cat.categories))]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scanpy as sc

# Define Okabe-Ito colorblind-safe palette
okabe_ito = [
    '#E69F00',  # orange
    '#56B4E9',  # sky blue
    '#009E73',  # green
    '#CC79A7',  # purple
    '#000000'   # black
]

# Assign to leiden clusters
n_clusters = adata1.obs['leiden'].nunique()
adata1.uns['leiden_colors'] = okabe_ito[:n_clusters]

# Set figure dimensions
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8

# Plot UMAP with custom colors
with plt.rc_context({
    "figure.figsize": (fig_width_inch, fig_height_inch),
    "figure.dpi": 600,
    "font.size": 5,
    "axes.labelsize": 5}):

    ax = sc.pl.umap(
        adata1,
        color=['leiden'],
        wspace=1,
        show=False,
        size=10,
        legend_loc='right margin'
    )

    ax.set_title('')

    for spine in ax.spines.values():
        spine.set_linewidth(0.3)

    plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/LM4445_leiden.svg', dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
#add Leiden to the slide
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scanpy as sc

# Define Okabe-Ito colorblind-safe palette
okabe_ito = [
    '#E69F00',  # orange
    '#56B4E9',  # sky blue
    '#009E73',  # green
    '#CC79A7',  # purple
    '#000000'   # black
]

# Assign to leiden clusters
n_clusters = adata1.obs['leiden'].nunique()
adata1.uns['leiden_colors'] = okabe_ito[:n_clusters]

# Set fixed figure dimensions in inches
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8

# Plot
with plt.rc_context({
    'figure.figsize': (fig_width_inch, fig_height_inch),
    'figure.dpi': 600,
    'font.size': 5
}):
    fig = sc.pl.spatial(
        adata1,
        color='leiden',        
        alpha=0.5,
        ncols=2,
        size=1,
        img_key='hires',
        show=False,
        alpha_img=0.5,
        bw=True,
        return_fig=True)

# Save
plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/11_samples/LM4445_leiden_slide.svg', dpi=600, bbox_inches='tight')
plt.show()